# Limpieza INE — Población histórica y superficie por municipio

Sigue el mismo patrón que el resto de notebooks de `notebooks/limpieza/` (`Limpieza_<fuente>.ipynb` → `DF_<FUENTE>_...csv`), pero con dos salidas en vez de una, porque son dos datasets con una granularidad distinta:

1. **Población histórica por municipio y año** (`DF_INE_POBLACION_HISTORICO.csv`) — una fila por `(municipio, año)`, con desglose por sexo. Sustituye al fichero estático `DF_INE_Poblaciones_España.xlsx` de un solo año que se usa en `Analisis_union.ipynb`.
2. **Superficie por municipio** (`DF_INE_SUPERFICIE_MUNICIPIOS.csv`) — una fila por municipio (no cambia con el año), pensada para calcular densidad de población más adelante.

**Fuentes**:
- Población: INE, operación ["Nomenclátor: Población por unidad poblacional"](https://www.ine.es/dyngs/INEbase/operacion.htm?c=Estadistica_C&cid=1254736177010&idp=1254735572981). Descarga de resultados: https://www.ine.es/dyngs/INEbase/operacion.htm?c=Estadistica_C&cid=1254736177010&menu=resultados&idp=1254735572981
- Superficie: IGN/CNIG, [Nomenclátor Geográfico de Municipios y Entidades de Población (NGMEP)](https://centrodedescargas.cnig.es/CentroDescargas/nomenclator-geografico-municipios-entidades-poblacion) — licencia CC-BY 4.0 (citar como "NGMEP CC-BY 4.0 Instituto Geográfico Nacional" en la memoria).

**Reutilizamos** `normalizar()` y el mapeo `PROVINCIA_INE` tal cual están en `Analisis_union.ipynb`, para que el cruce de nombres de municipio/provincia sea consistente en todo el proyecto. Si se corrige algo en esa función allí, hay que replicarlo aquí (no hay un módulo `.py` compartido entre notebooks todavía).


In [1]:
import zipfile
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests

## Utilidades compartidas con `Analisis_union.ipynb`

Copiadas literalmente de allí para no depender de un import cruzado entre notebooks.


In [2]:
def normalizar(texto):
    """Mayúsculas y sin acentos, para poder cruzar texto libre con el nomenclátor del INE."""
    if pd.isna(texto):
        return None
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return texto.upper().strip()

# Las comunidades multiprovinciales (Galicia, Catalunya, Andalucía, la
# Comunidad Valenciana, Castilla y León, Castilla-La Mancha, Aragón) no se
# pueden asignar a una sola provincia del INE, así que quedan sin cruzar.
PROVINCIA_INE = {
    1: ["Álava", "Araba", "Araba/Álava"], 2: ["Albacete"], 3: ["Alicante", "Alacant"],
    4: ["Almería"], 5: ["Ávila"], 6: ["Badajoz"], 7: ["Baleares", "Illes Balears", "Baleares (Illes)"],
    8: ["Barcelona"], 9: ["Burgos"], 10: ["Cáceres"], 11: ["Cádiz"], 12: ["Castellón", "Castelló"],
    13: ["Ciudad Real"], 14: ["Córdoba"], 15: ["La Coruña", "A Coruña"], 16: ["Cuenca"],
    17: ["Girona", "Gerona"], 18: ["Granada"], 19: ["Guadalajara"], 20: ["Guipúzcoa", "Gipuzkoa"],
    21: ["Huelva"], 22: ["Huesca"], 23: ["Jaén"], 24: ["León"], 25: ["Lleida", "Lérida"],
    26: ["La Rioja"], 27: ["Lugo"], 28: ["Madrid", "Comunidad de Madrid"], 29: ["Málaga"],
    30: ["Murcia", "Región de Murcia"], 31: ["Navarra"], 32: ["Ourense", "Orense"],
    33: ["Asturias", "Asturias / Asturies"], 34: ["Palencia"], 35: ["Las Palmas"],
    36: ["Pontevedra"], 37: ["Salamanca"], 38: ["Santa Cruz de Tenerife"], 39: ["Cantabria"],
    40: ["Segovia"], 41: ["Sevilla"], 42: ["Soria"], 43: ["Tarragona"], 44: ["Teruel"],
    45: ["Toledo"], 46: ["Valencia", "València"], 47: ["Valladolid"], 48: ["Vizcaya", "Bizkaia"],
    49: ["Zamora"], 50: ["Zaragoza"], 51: ["Ceuta"], 52: ["Melilla"],
}
PROVINCIA_A_CODIGO = {normalizar(n): c for c, ns in PROVINCIA_INE.items() for n in ns}

In [3]:
# Ajustar si el notebook se ejecuta desde otra ubicación: se asume que se
# lanza desde notebooks/limpieza/, con data/ dos niveles por encima.
# Se usa "INE" en mayúsculas para que coincida con la carpeta data/raw/INE
# que ya existe en el proyecto (con Nacional_2024.zip, Provincia08.xlsx y
# BD_Municipios-Entidades.zip ya descargados).
BASE_DIR = Path("../..")
RAW_DIR = BASE_DIR / "data" / "raw" / "INE"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "INE"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Población histórica por municipio y año

El INE publica un ZIP nacional por año en `https://www.ine.es/nomen_files/nacional/es/Nacional_{AÑO}.zip`. **El contenido del ZIP no es idéntico en todos los años** (comprobado descargando y mirando dentro de varios):

- **2024 en adelante**: 4 ficheros — `nomdefAAAA.xlsx` (multi-hoja: Sexo/Nacionalidad/Edad) + 3 `.txt` de ancho fijo ya separados por desglose (`..._sex.txt`, `..._edad.txt`, `..._naci.txt`).
- **~2000–2023**: 2 ficheros — un `.txt` (o sin extensión) de ancho fijo que ya trae Total/Hombres/Mujeres juntos, + un `.xlsx` con una sola hoja.
- **1981/1991/1996/1998/1999**: no comprobado en detalle todavía; si se necesitan carreras de esos años habrá que revisar el ZIP a mano y ajustar `localizar_fichero_poblacion()` más abajo.

En vez de leer los `.xlsx` (hasta 16 MB descomprimidos cada uno) leemos directamente el `.txt` de ancho fijo, mucho más ligero. El formato de cada línea es estable en todos los años comprobados:

```
AAMMMUUUUUU NOMBRE (con relleno)...        TOTAL  HOMBRES  MUJERES
```

donde `AA` = provincia (2 dígitos), `MMM` = municipio (3 dígitos), `UUUUUU` = unidad poblacional (6 dígitos; `000000` = total del municipio, el resto son entidades/núcleos/diseminados que descartamos, igual que en `Analisis_union.ipynb`), y los tres últimos números ocupan **7 caracteres cada uno, alineados a la derecha** (con ceros... con espacios de relleno, no ceros).

**Importante — no vale con partir por espacios en blanco**: para municipios grandes (Madrid: 3.422.416 habitantes) el número ocupa los 7 caracteres completos y por tanto los tres campos quedan pegados sin ningún espacio de separación (`...342241616029241819492`). Lo comprobé descargando el ZIP de 2024 real: con una regex basada en espacios, las 3 líneas de Madrid no parseaban y su población **desaparecía silenciosamente** del total (48,6 M según el INE → 45,2 M sin Madrid). Por eso el parseo de abajo corta por posición fija (últimos 21 caracteres = 3 campos de 7), no por espacios — con esto la suma de población 2024 cuadra exactamente con la cifra oficial del INE.

**Encoding**: los ficheros de 2024 en adelante están en UTF-8; los de años anteriores (comprobado en 2000 y 2013) están en Latin-1 — si se fuerza UTF-8 aparecen caracteres corruptos en nombres con acentos/ñ. Lo detectamos por año en vez de asumir uno fijo.


In [4]:
def descargar_zip_nacional(anyo: int) -> Path:
    """Descarga (si no existe ya) el ZIP nacional del Nomenclátor para `anyo`."""
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    ruta = RAW_DIR / f"Nacional_{anyo}.zip"
    if ruta.exists():
        return ruta
    url = f"https://www.ine.es/nomen_files/nacional/es/Nacional_{anyo}.zip"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    ruta.write_bytes(r.content)
    return ruta


def localizar_fichero_poblacion(zf: zipfile.ZipFile, anyo: int) -> str:
    """Dentro del ZIP de un año, localiza el .txt con Total/Hombres/Mujeres."""
    nombres = zf.namelist()
    # Esquema 2024+: viene ya separado, buscamos el de sexo explícitamente.
    candidatos = [n for n in nombres if "sex" in n.lower() and n.lower().endswith(".txt")]
    if candidatos:
        return candidatos[0]
    # Esquema anterior: un único fichero de texto (a veces sin extensión)
    # que ya trae los tres desgloses juntos.
    candidatos = [n for n in nombres if not n.lower().endswith(".xlsx")]
    if len(candidatos) == 1:
        return candidatos[0]
    raise ValueError(f"[{anyo}] no he podido identificar el fichero de población entre: {nombres}")


def decodificar(datos: bytes) -> str:
    """UTF-8 en años recientes, Latin-1 en años antiguos."""
    try:
        return datos.decode("utf-8")
    except UnicodeDecodeError:
        return datos.decode("latin-1")

In [5]:
# Provincia (2) + municipio (3) + unidad poblacional (6) = 11 caracteres fijos,
# y los últimos 3*7=21 caracteres de la línea son Total/Hombres/Mujeres (7 cada
# uno, alineados a la derecha). Todo lo que queda en medio es el nombre.
# NO se puede partir por espacios en blanco: ver la nota de más arriba sobre Madrid.
ANCHO_CAMPO_NUMERICO = 7
LARGO_CABECERA = 11  # provincia + municipio + unidad poblacional


def parsear_linea(linea: str):
    largo_minimo = LARGO_CABECERA + 3 * ANCHO_CAMPO_NUMERICO
    if len(linea) < largo_minimo:
        return None
    prov, muni, unidad = linea[0:2], linea[2:5], linea[5:11]
    resto = linea[11:]
    bloque_numeros = resto[-3 * ANCHO_CAMPO_NUMERICO:]
    nombre = resto[: -3 * ANCHO_CAMPO_NUMERICO]
    try:
        total = int(bloque_numeros[0:ANCHO_CAMPO_NUMERICO])
        hombres = int(bloque_numeros[ANCHO_CAMPO_NUMERICO:2 * ANCHO_CAMPO_NUMERICO])
        mujeres = int(bloque_numeros[2 * ANCHO_CAMPO_NUMERICO:3 * ANCHO_CAMPO_NUMERICO])
    except ValueError:
        return None
    return prov, muni, unidad, nombre.strip(" \t"), total, hombres, mujeres


def parsear_anyo(anyo: int, ruta_zip: Path) -> pd.DataFrame:
    with zipfile.ZipFile(ruta_zip) as zf:
        nombre_entrada = localizar_fichero_poblacion(zf, anyo)
        texto = decodificar(zf.read(nombre_entrada))

    filas = []
    sin_parsear = 0
    for linea in texto.splitlines():
        if not linea.strip():
            continue
        r = parsear_linea(linea)
        if r is None:
            sin_parsear += 1
            continue
        prov, muni, unidad, nombre, total, hombres, mujeres = r
        if unidad != "000000":
            continue  # solo el total municipal, igual que en Analisis_union.ipynb
        filas.append({
            "provincia_ine": int(prov),
            "codigo_municipio": muni,
            "nombre_ine": nombre,
            "total_poblacion": total,
            "poblacion_h": hombres,
            "poblacion_d": mujeres,
            "anyo": anyo,
        })
    if sin_parsear:
        print(f"[{anyo}] aviso: {sin_parsear} líneas no encajan con el formato esperado (revisar a mano)")

    df = pd.DataFrame(filas)
    df["municipio_norm"] = df["nombre_ine"].apply(normalizar)
    return df

In [6]:
# 2000-2025: rango continuo y es el que más probablemente cubre las carreras
# del dataset. Los años sueltos anteriores (1981/1991/1996/1998/1999) no se
# incluyen por defecto -- añadirlos a esta lista si hacen falta, revisando antes
# el contenido de su ZIP (ver nota más arriba).
ANYOS = list(range(2000, 2026))

dfs_por_anyo = []
for anyo in ANYOS:
    try:
        ruta_zip = descargar_zip_nacional(anyo)
        dfs_por_anyo.append(parsear_anyo(anyo, ruta_zip))
    except Exception as e:
        print(f"[{anyo}] ERROR: {e}")

poblacion_historico = pd.concat(dfs_por_anyo, ignore_index=True)
poblacion_historico = poblacion_historico[
    ["municipio_norm", "provincia_ine", "anyo", "total_poblacion", "poblacion_h", "poblacion_d",
     "nombre_ine", "codigo_municipio"]
]
poblacion_historico.head()

,municipio_norm,provincia_ine,anyo,total_poblacion,poblacion_h,poblacion_d,nombre_ine,codigo_municipio
0,ALEGRIA-DULANTZI,1,2000,1401,731,670,ALEGRIA-DULANTZI,001
1,AMURRIO,1,2000,9720,4839,4881,AMURRIO,002
2,ARAMAIO,1,2000,1416,747,669,ARAMAIO,003
3,ARTZINIEGA,1,2000,1338,669,669,ARTZINIEGA,004
4,ARMINON,1,2000,146,76,70,ARMIÑON,006


### Control de calidad rápido


In [7]:
print(f"Filas totales: {len(poblacion_historico)}")
print(f"Años cubiertos: {sorted(poblacion_historico['anyo'].unique())}")
print()
print("Filas por año (debería rondar el nº de municipios españoles, ~8100):")
print(poblacion_historico.groupby("anyo").size())
print()
print("Filas con población total nula o <= 0 por año (si hay muchas, revisar el parseo de ese año):")
print(poblacion_historico[poblacion_historico["total_poblacion"].fillna(0) <= 0].groupby("anyo").size())
print()
duplicados = poblacion_historico.duplicated(subset=["municipio_norm", "provincia_ine", "anyo"]).sum()
print(f"Duplicados por (municipio_norm, provincia_ine, anyo): {duplicados}")
print()
# Chequeo de bulto: la población total de España ronda los 47-49 millones en
# todo este rango de años. Si algún año se aleja mucho, es señal de que el
# parseo de ancho fijo ha fallado para ese año (p.ej. municipios muy grandes
# como Madrid, cuyos 3 campos numéricos quedan pegados sin espacios -- ver la
# nota más arriba). Aquí es donde se habría visto el problema con Madrid.
print("Población total de España por año (contraste con la cifra oficial del INE):")
print(poblacion_historico.groupby("anyo")["total_poblacion"].sum())

Filas totales: 211081
Años cubiertos: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Filas por año (debería rondar el nº de municipios españoles, ~8100):
anyo
2000    8104
2001    8107
2002    8108
2003    8108
2004    8109
2005    8109
2006    8110
2007    8111
2008    8112
2009    8112
2010    8114
2011    8116
2012    8116
2013    8117
2014    8117
2015    8119
2016    8125
2017    8124
2018    8124
2019    8131
2020    8131
2021    8131
2022    8131
2023    8131
2024    8132
2025    8132
dtype: int64

Filas con población total nula o <= 0 por año (si hay muchas, revisar el parseo de ese año):
Series([], dtype: int64)

Duplicados por (municipio_norm, provincia_ine, anyo): 0

Población total de España por año (contraste con la cifra oficial del INE):
anyo
2000    40499791
2001    41116842
2002    41837894
2003    42717064
2004    43197684
2005    44108530
2006    44708964
20

In [8]:
salida_poblacion = PROCESSED_DIR / "DF_INE_POBLACION_HISTORICO.csv"
poblacion_historico.to_csv(salida_poblacion, index=False, encoding="utf-8")
print(f"Guardado en {salida_poblacion.resolve()}")

Guardado en /Users/claudiarm2002/Desktop/TFM/data/processed/INE/DF_INE_POBLACION_HISTORICO.csv


## 2. Superficie por municipio (para densidad de población)

Fuente: [NGMEP (IGN/CNIG)](https://centrodedescargas.cnig.es/CentroDescargas/nomenclator-geografico-municipios-entidades-poblacion), licencia CC-BY 4.0 (citar como "NGMEP CC-BY 4.0 Instituto Geográfico Nacional" en la memoria). Es un dato prácticamente estático (los límites municipales casi no cambian de un año a otro), así que aquí generamos **una sola tabla, sin desglose por año** — a diferencia de la población.

Ya tienes el ZIP descargado en `data/raw/INE/BD_Municipios-Entidades.zip` (el botón "Descargar" del Centro de Descargas del CNIG no es un enlace directo -- dispara dos peticiones POST ligadas a la sesión del navegador, por eso no lo automatizamos con `requests` como los ZIP de población; pero al ser un fichero que se baja una vez, la descarga manual que ya hiciste es suficiente). Dentro trae, entre otros, `MUNICIPIOS.csv` (`;` como separador, encoding Latin-1), con las columnas `COD_INE` (11 dígitos: 2 de provincia + 3 de municipio + 6 de unidad, mismo esquema que en la sección 1), `NOMBRE_ACTUAL` y `SUPERFICIE` -- esta última en **hectáreas y con coma decimal** (`"1994,5872"`), hay que convertirla antes de pasar a km².


In [9]:
NGMEP_ZIP = RAW_DIR / "BD_Municipios-Entidades.zip"

with zipfile.ZipFile(NGMEP_ZIP) as zf:
    with zf.open("MUNICIPIOS.csv") as f:
        superficie = pd.read_csv(f, sep=";", encoding="latin-1", dtype={"COD_INE": str})

superficie.head()

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD
0,01001000000,1010014,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,0113-3,"-2,512507724","42,84045247",Detección automática,568,MDT
1,01002000000,1010029,1020,1,Araba/Álava,Amurrio,10346,"9617,86",65701,1002000201,Amurrio,9256,0086-4,"-3,001015194","43,05265767",Detección automática,217,MDT
2,01003000000,1010035,1030,1,Araba/Álava,Aramaio,1353,"7308,96",42097,1003000601,Ibarra,731,0087-4,"-2,564829379","43,05257873",Detección automática,325,MDT
3,01004000000,1010040,1040,1,Araba/Álava,Artziniega,1868,"2728,73",22886,1004000101,Artziniega,1732,0086-1,"-3,13052099","43,1217919",Detección automática,199,MDT
4,01006000000,1010066,1060,1,Araba/Álava,Armiñón,233,"1297,27",24707,1006000101,Armiñón,106,0137-4,"-2,872270813","42,72340924",Detección automática,466,MDT


In [10]:
# COD_INE: 2 dígitos de provincia + 3 de municipio + 6 de unidad poblacional
# ("000000" = municipio) -> mismo criterio de cruce que en la sección 1
# (municipio_norm + provincia_ine).
cod_ine = superficie["COD_INE"].str.zfill(11)
superficie["provincia_ine"] = cod_ine.str.slice(0, 2).astype(int)
superficie["codigo_municipio"] = cod_ine.str.slice(2, 5)
superficie["municipio_norm"] = superficie["NOMBRE_ACTUAL"].apply(normalizar)

# Hectáreas con coma decimal (formato español) -> km2 con punto decimal.
superficie["superficie_km2"] = (
    superficie["SUPERFICIE"].astype(str).str.replace(",", ".", regex=False).astype(float) / 100
)

superficie_municipios = superficie[
    ["municipio_norm", "provincia_ine", "codigo_municipio", "superficie_km2"]
].drop_duplicates()

print(f"Municipios con superficie: {len(superficie_municipios)}")
print(superficie_municipios['superficie_km2'].describe())
superficie_municipios.head()

Municipios con superficie: 8132
count    8132.000000
mean       62.074400
std        92.006215
min         0.012600
25%        18.404896
50%        34.877865
75%        68.895419
max      1750.229100
Name: superficie_km2, dtype: float64


,municipio_norm,provincia_ine,codigo_municipio,superficie_km2
0,ALEGRIA-DULANTZI,1,001,19.945872
1,AMURRIO,1,002,96.178600
2,ARAMAIO,1,003,73.089600
3,ARTZINIEGA,1,004,27.287300
4,ARMINON,1,006,12.972700


In [11]:
salida_superficie = PROCESSED_DIR / "DF_INE_SUPERFICIE_MUNICIPIOS.csv"
superficie_municipios.to_csv(salida_superficie, index=False, encoding="utf-8")
print(f"Guardado en {salida_superficie.resolve()}")

Guardado en /Users/claudiarm2002/Desktop/TFM/data/processed/INE/DF_INE_SUPERFICIE_MUNICIPIOS.csv


## Notas para el notebook que consuma estos ficheros

- **Cruce**: mismo criterio que ya usa `Analisis_union.ipynb` — `(municipio_norm, provincia_ine)`. `DF_INE_POBLACION_HISTORICO.csv` añade la columna `anyo`, así que el cruce con las carreras pasa a ser `(municipio_norm, provincia_ine, anyo)`.
- **Años sin padrón (p. ej. 2026, el año en curso)**: aplicar el mismo fallback que ya se usa en el proyecto — quedarse con el último año disponible (2025) en vez de dejar el cruce a `NaN`.
- **Densidad**: una vez cruzados los dos ficheros por `(municipio_norm, provincia_ine)` — la superficie no lleva `anyo` —, `densidad = total_poblacion / superficie_km2`.
- **Comunidades multiprovinciales**: igual que en `Analisis_union.ipynb`, las filas de carreras cuya `provincia` es en realidad una comunidad autónoma (Galicia, Catalunya, etc.) no cruzan ni aquí ni allí — no es un problema nuevo de este notebook.
- **Pendiente**: revisar el formato de los ZIP de 1981/1991/1996/1998/1999 si se necesitan carreras de esos años. Si el NGMEP se vuelve a descargar más adelante (por ejemplo el año que viene) y cambia de nombre de fichero, actualizar `NGMEP_ZIP` en la sección 2.
